<a href="https://colab.research.google.com/github/ml-project-sv/Cross-Platform-Binary-Code-Similarity-Detection/blob/main/model_experiment_deepsets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import os, glob, json
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

In [38]:
PATH = '/content/Cross-Platform-Binary-Code-Similarity-Detection'

# Wandb Login

In [39]:
import wandb
wandb.login()

True

# Arch-Opt Parsing Function

In [40]:
_ARCHES = ('x86', 'arm32', 'arm', 'mips32', 'mips')
_OPTS   = ('O0', 'O1', 'O2', 'O3')

In [41]:
def parse_arch_opt(src):
  arch = next((a for a in _ARCHES if f'-{a}-' in src), None)
  opt  = next((o for o in _OPTS   if f'-{o}' in src), None)
  return arch, opt

# Load Dataset

In [42]:
def load_dataset(dirs, min_blocks=4, n_feat=8):
  if isinstance(dirs, str):
    dirs = [dirs]

  files = []
  for d in dirs:
    files += glob.glob(os.path.join(d, '*.json'))

  if not files:
    raise FileNotFoundError(f'no .json under {dirs}')

  funcs, fnames, dropped = [], [], 0
  for fp in files:
    with open(fp) as fh:
      for line in fh:
        line = line.strip()
        if not line: continue

        r = json.loads(line)
        feats = r['features']
        if len(feats) < min_blocks:
          dropped += 1
          continue

        r['X'] = np.asarray(feats, dtype=np.float32)[:, :n_feat]
        r['succs'] = r['succs'] if 'succs' in r else None
        r['arch'], r['opt'] = parse_arch_opt(r.get('src', ''))

        funcs.append(r)
        fnames.append(r['fname'])

  uniq = sorted(set(fnames))
  classes = {n: i for i, n in enumerate(uniq)}
  labels = np.array([classes[n] for n in fnames], dtype=np.int64)
  print(f'[load] files={len(files)} funcs={len(funcs)} classes={len(uniq)} dropped(<{min_blocks} blocks)={dropped} n_feat={n_feat}')

  return funcs, labels, classes


# Split Dataset

In [43]:
def make_split(labels, ratios=(.8, .1, .1), seed=1337):
  n_cls = labels.max() + 1
  rng = np.random.default_rng(seed)
  cls_perm = rng.permutation(n_cls)

  n_train = int(ratios[0] * n_cls)
  n_val   = int(ratios[1] * n_cls)

  train_cls = set(cls_perm[:n_train].tolist())
  val_cls   = set(cls_perm[n_train: n_train + n_val].tolist())
  test_cls  = set(cls_perm[n_train + n_val:].tolist())

  train_mask = np.array([c in train_cls for c in labels])
  val_mask   = np.array([c in val_cls   for c in labels])
  test_mask  = np.array([c in test_cls  for c in labels])

  print(f'[split] train={train_mask.sum()} val={val_mask.sum()} test={test_mask.sum()}')
  return train_mask, val_mask, test_mask

 # Read Dataset

In [44]:
def make_funcs_labels(path, min_blocks, n_feat, ratios):
  funcs, labels, classes = load_dataset(path, min_blocks=min_blocks, n_feat=n_feat)
  train_mask, val_mask, test_mask = make_split(labels, ratios=ratios)

  train_funcs = [f for f, m in zip(funcs, train_mask) if m]
  val_funcs   = [f for f, m in zip(funcs, val_mask) if m]
  test_funcs  = [f for f, m in zip(funcs, test_mask) if m]

  train_labels = labels[train_mask]
  val_labels   = labels[val_mask]
  test_labels  = labels[test_mask]

  return train_funcs, val_funcs, test_funcs, train_labels, val_labels, test_labels

# Functions for Metrics

In [45]:
def _normalize(V, eps=1e-10):
  return V / (np.linalg.norm(V, axis=1, keepdims=True) + eps)

In [46]:
def _group_by_class(labels):
  d = {}
  for i, c in enumerate(labels):
    d.setdefault(int(c), []).append(i)
  return d

In [47]:
# axis query and target belong to
def _axis(qa, qo, ta, to):
  if qa == ta and qo != to: return 'XO'
  if qa != ta and qo == to: return 'XA'
  if qa != ta and qo != to: return 'XM'
  return 'same'

# Metrics

### Auc Metrics

In [48]:
def auc_pairs(E, score_fn, labels, n_pairs=8000, seed=1337):
  rng = np.random.default_rng(seed);
  by = _group_by_class(labels)

  # leave functions which have at least two compiled versions (eligible for positive pairs)
  pos_c = [c for c, v in by.items() if len(v) >= 2]

  s, y = [], []

  # same functions with different arch-opt pair
  for _ in range(n_pairs // 2):
    c = pos_c[rng.integers(len(pos_c))]
    i, j = rng.choice(by[c], 2, replace=False)
    s.append(float(score_fn(E, i, j)))
    y.append(1)

  # different functions
  n = len(labels); got = 0
  while got < n_pairs // 2:
    i, j = rng.integers(n), rng.integers(n)
    if labels[i] == labels[j]:
      continue
    s.append(float(score_fn(E, i, j)))
    y.append(0)
    got += 1

  return s, y

### Retrieval Metrics

In [49]:
def retrieval(E, score_fn, labels, arches, opts, pool_sizes=(10, 100, 1000), n_queries=1000, ks=(1, 5, 10), seed=1337, per_axis=True):
  rng = np.random.default_rng(seed)
  by = _group_by_class(labels)

  pos_c = [c for c, v in by.items() if len(v) >= 2]
  axes = ['XO', 'XA', 'XM'] if per_axis else []
  out = {}

  for pool in pool_sizes:
    if pool > len(labels): continue

    # init recall hit for each top k
    rec = {k: 0 for k in ks}

    # init running sum of 1/rank
    mrr = 0.0

    # init number of queries processed
    Q = 0

    # init same metrics but for each axis now
    arec = {ax: {k: 0 for k in ks} for ax in axes}
    amrr = {ax: 0.0 for ax in axes}
    acnt = {ax: 0 for ax in axes}


    for _ in range(n_queries):
      # choose class and positive pairs of functions
      c = pos_c[rng.integers(len(pos_c))]
      q, tgt = rng.choice(by[c], 2, replace=False)

      # choosing functions with distinct classes
      dist = []
      while len(dist) < pool - 1:
        d = rng.integers(len(labels))
        if labels[d] != c:
          dist.append(d)

      cand = np.array([tgt] + dist)
      sims = score_fn(E, q, cand)

      # calculate where positive pair is ranked
      rank = np.where(np.argsort(-sims) == 0)[0][0] + 1

      # update metrics
      mrr += 1.0 / rank
      for k in ks:
        if rank <= k: rec[k] += 1
      Q += 1

      # if per_axis is enabled, log same metrics but separately for each (q, tgt) axis
      if per_axis:
        ax = _axis(arches[q], opts[q], arches[tgt], opts[tgt])
        if ax in arec:
          amrr[ax] += 1.0 / rank; acnt[ax] += 1
          for k in ks:
            if rank <= k: arec[ax][k] += 1

    # save results to out
    for k in ks:
      out[f'recall@{k}_pool{pool}'] = rec[k] / Q
    out[f'mrr_pool{pool}'] = mrr / Q

    for ax in axes:
      if acnt[ax] == 0:
        continue
      for k in ks:
        out[f'{ax}_recall@{k}_pool{pool}'] = arec[ax][k] / acnt[ax]
      out[f'{ax}_mrr_pool{pool}'] = amrr[ax] / acnt[ax]

  return out

# Evaluate

In [50]:
def evaluate(embedder, score_fn, funcs, labels, pool_sizes=(10, 100, 1000), n_pairs=8000, n_queries=500, ks=(1, 5, 10), per_axis=True):
  E = embedder.embed_batch(funcs)
  arches = [f.get('arch') for f in funcs]
  opts   = [f.get('opt')  for f in funcs]

  # roc and pr
  s, y = auc_pairs(E, score_fn, labels, n_pairs=n_pairs)
  roc_auc, pr_auc = roc_auc_score(y, s), average_precision_score(y, s)

  # retrieval
  retr = retrieval(E, score_fn, labels, arches, opts, pool_sizes=pool_sizes, n_queries=n_queries, ks=ks, per_axis=per_axis)

  return  {'roc_auc': roc_auc, 'pr_auc': pr_auc, 's': s, 'y': y, **retr}

# Printing & Plotting & Logging Data

### Plotting

In [51]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve

AXIS_LABELS = {'XO': 'XO cross-opt', 'XA': 'XA cross-arch', 'XM': 'XM cross-both'}

def _pools_in(r, pools):
    return [p for p in pools if f'recall@1_pool{p}' in r]

def make_plots(results, axes, pools, ks, model_name='model'):
    figs = {}
    tags = list(results.keys())
    C = plt.cm.tab10(np.linspace(0, 1, 10))

    # ROC curves
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, (tag, r) in enumerate(results.items()):
        if 's' in r and 'y' in r:
            fpr, tpr, _ = roc_curve(r['y'], r['s'])
            ax.plot(fpr, tpr, color=C[i], label=f"{tag} (AUC={r['roc_auc']:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', alpha=.4, label='random')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_name}: ROC curves'); ax.legend(loc='lower right'); ax.grid(alpha=.3)
    figs['roc_curves'] = fig

    # Precision-Recall
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, (tag, r) in enumerate(results.items()):
        if 's' in r and 'y' in r:
            prec, rec, _ = precision_recall_curve(r['y'], r['s'])
            ax.plot(rec, prec, color=C[i], label=f"{tag} (AP={r['pr_auc']:.3f})")
    ax.axhline(0.5, ls='--', color='k', alpha=.4, label='random (balanced)')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'{model_name}: Precision-Recall curves'); ax.legend(loc='lower left'); ax.grid(alpha=.3)
    figs['pr_curves'] = fig

    # Recall@1 vs pool size
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (tag, r) in enumerate(results.items()):
        ps = _pools_in(r, pools); ys = [r[f'recall@1_pool{p}'] for p in ps]
        ax.plot(ps, ys, marker='o', color=C[i], label=tag)
    ax.set_xscale('log'); ax.set_xlabel('pool size'); ax.set_ylabel('Recall@1')
    ax.set_title(f'{model_name}: Retrieval difficulty (R@1 vs pool)')
    ax.legend(); ax.grid(alpha=.3)
    figs['recall1_vs_pool'] = fig

    # MRR vs pool size
    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (tag, r) in enumerate(results.items()):
        ps = _pools_in(r, pools); ys = [r[f'mrr_pool{p}'] for p in ps]
        ax.plot(ps, ys, marker='s', color=C[i], label=tag)
    ax.set_xscale('log'); ax.set_xlabel('pool size'); ax.set_ylabel('MRR')
    ax.set_title(f'{model_name}: MRR vs pool'); ax.legend(); ax.grid(alpha=.3)
    figs['mrr_vs_pool'] = fig

    # Per-axis R@1 grouped bars
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(axes)); w = 0.8 / max(len(tags), 1)
    for i, (tag, r) in enumerate(results.items()):
        p = max(_pools_in(r, pools)); ys = [r.get(f'{a}_recall@1_pool{p}', 0) for a in axes]
        ax.bar(x + i*w, ys, w, color=C[i], label=f'{tag} (pool{p})')
    ax.set_xticks(x + w*(len(tags)-1)/2)
    ax.set_xticklabels([AXIS_LABELS.get(a, a) for a in axes])
    ax.set_ylabel('Recall@1'); ax.set_title(f'{model_name}: Per-axis difficulty (XO > XA > XM)')
    ax.legend(); ax.grid(alpha=.3, axis='y')
    figs['per_axis_bars'] = fig

    # Recall@k by axis
    main = tags[0]; r = results[main]
    p = 100 if f'XO_recall@1_pool100' in r else max(_pools_in(r, pools))
    fig, ax = plt.subplots(figsize=(6, 4))
    for a in axes:
        ys = [r.get(f'{a}_recall@{k}_pool{p}') for k in ks]
        if all(v is not None for v in ys):
            ax.plot(ks, ys, marker='o', label=AXIS_LABELS.get(a, a))
    ax.set_xlabel('k'); ax.set_ylabel(f'Recall@k @ pool{p}'); ax.set_xticks(ks)
    ax.set_title(f'{model_name}: Recall@k by axis ({main})'); ax.legend(); ax.grid(alpha=.3)
    figs['recallk_by_axis'] = fig

    # Heatmap: axis x pool
    r = results[main]; ps = _pools_in(r, pools)
    fig, ax = plt.subplots(figsize=(5, 4))
    M = np.array([[r.get(f'{a}_recall@1_pool{p}', np.nan) for p in ps] for a in axes])
    im = ax.imshow(M, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(ps))); ax.set_xticklabels(ps)
    ax.set_yticks(range(len(axes))); ax.set_yticklabels(axes)
    ax.set_xlabel('pool'); ax.set_ylabel('axis'); ax.set_title(f'{model_name}: R@1 heatmap ({main})')
    for ii in range(len(axes)):
        for jj in range(len(ps)):
            if not np.isnan(M[ii, jj]):
                ax.text(jj, ii, f'{M[ii,jj]:.2f}', ha='center', va='center', color='w', fontsize=9)
    fig.colorbar(im); figs['heatmap_axis_pool'] = fig

    # ROC-AUC & PR-AUC bars
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(tags)); w = 0.35
    ax.bar(x - w/2, [results[t]['roc_auc'] for t in tags], w, label='ROC-AUC')
    ax.bar(x + w/2, [results[t]['pr_auc'] for t in tags], w, label='PR-AUC')
    ax.set_xticks(x); ax.set_xticklabels(tags, rotation=20, ha='right'); ax.set_ylim(0.5, 1.0)
    ax.set_ylabel('score'); ax.set_title(f'{model_name}: AUC & PR-AUC'); ax.legend(); ax.grid(alpha=.3, axis='y')
    figs['auc_prauc_bars'] = fig

    # Score distribution (pos vs neg)
    r = results[main]
    if 's' in r and 'y' in r:
        s = np.array(r['s']); y = np.array(r['y'])
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(s[y == 1], bins=40, alpha=.6, label='same function (pos)', color='green', density=True)
        ax.hist(s[y == 0], bins=40, alpha=.6, label='different (neg)', color='red', density=True)
        ax.set_xlabel('similarity score'); ax.set_ylabel('density')
        ax.set_title(f'{model_name}: Score separation ({main})'); ax.legend(); ax.grid(alpha=.3)
        figs['score_distribution'] = fig


    # Recall@k coverage curve (per pool)
    main = tags[0]; r = results[main]
    fig, ax = plt.subplots(figsize=(6, 4))
    for p in _pools_in(r, pools):
        ys = [r.get(f'recall@{k}_pool{p}') for k in ks]
        if all(v is not None for v in ys):
            ax.plot(ks, ys, marker='o', label=f'pool{p}')
    ax.set_xlabel('k'); ax.set_ylabel('Recall@k'); ax.set_xticks(ks)
    ax.set_title(f'{model_name}: Recall@k coverage ({main})')
    ax.legend(); ax.grid(alpha=.3)
    figs['recallk_coverage'] = fig


    # AUC vs retrieval gap
    for k in ks:
        fig, ax = plt.subplots(figsize=(6, 4))
        x = np.arange(len(tags)); w = 0.35
        ax.bar(x - w/2, [results[t]['roc_auc'] for t in tags], w,
               label='ROC-AUC', color='steelblue')
        # Recall@k at the largest pool available for each eval set
        rk_key = lambda t: f'recall@{k}_pool{max(_pools_in(results[t], pools))}'
        ax.bar(x + w/2, [results[t].get(rk_key(t), 0) for t in tags], w,
               label=f'R@{k} (largest pool)', color='coral')
        ax.set_xticks(x); ax.set_xticklabels(tags, rotation=20, ha='right'); ax.set_ylim(0, 1)
        ax.set_ylabel('score'); ax.set_title(f'{model_name}: AUC vs R@{k} gap')
        ax.legend(); ax.grid(alpha=.3, axis='y')
        figs[f'auc_retrieval_gap_k{k}'] = fig


    # MRR heatmap (axis x pool)
    r = results[main]; ps = _pools_in(r, pools)
    fig, ax = plt.subplots(figsize=(5, 4))
    M = np.array([[r.get(f'{a}_mrr_pool{p}', np.nan) for p in ps] for a in axes])
    im = ax.imshow(M, cmap='magma', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(ps))); ax.set_xticklabels(ps)
    ax.set_yticks(range(len(axes))); ax.set_yticklabels(axes)
    ax.set_xlabel('pool'); ax.set_ylabel('axis')
    ax.set_title(f'{model_name}: MRR heatmap ({main})')
    for ii in range(len(axes)):
        for jj in range(len(ps)):
            if not np.isnan(M[ii, jj]):
                ax.text(jj, ii, f'{M[ii,jj]:.2f}', ha='center', va='center', color='w', fontsize=9)
    fig.colorbar(im); figs['mrr_heatmap'] = fig


    # Score CDF: cumulative distribution of pos vs neg similarity scores.
    r = results[main]
    if 's' in r and 'y' in r:
        s = np.array(r['s']); y = np.array(r['y'])
        fig, ax = plt.subplots(figsize=(6, 4))
        for lab, mask, col in [('same function (pos)', y == 1, 'green'),
                               ('different (neg)', y == 0, 'red')]:
            v = np.sort(s[mask]); cdf = np.arange(1, len(v)+1) / len(v)
            ax.plot(v, cdf, color=col, label=lab)
        ax.set_xlabel('similarity score'); ax.set_ylabel('cumulative fraction')
        ax.set_title(f'{model_name}: Score CDF ({main})')
        ax.legend(); ax.grid(alpha=.3)
        figs['score_cdf'] = fig

    return figs

### Printing

In [52]:
def print_comparison(results):
    metrics = [('ROC-AUC','roc_auc'), ('R@1@1000','recall@1_pool1000'),
               ('MRR@1000','mrr_pool1000'), ('XO@1@1k','XO_recall@1_pool1000'),
               ('XA@1@1k','XA_recall@1_pool1000'), ('XM@1@1k','XM_recall@1_pool1000')]
    tags = list(results.keys())
    print(f"\n{'metric':>12} | " + " | ".join(f"{t:>14}" for t in tags))
    print("-" * (14 + 3 + len(tags)*17))
    for label, key in metrics:
        row = f"{label:>12} | "
        row += " | ".join(f"{results[t].get(key, float('nan')):>14.4f}" for t in tags)
        print(row)

### Wandb Logging

In [53]:
def _cast(v):
    return float(v) if isinstance(v, (np.floating, np.integer, np.bool_)) else v

def log(run, results, figs, table_name='metrics'):
    # log metrics
    for tag, res in results.items():
        payload = {f'{tag}/{k}': _cast(v) for k, v in res.items()}
        run.log(payload)
        for k, v in payload.items():
            run.summary[k] = v

    table = wandb.Table(columns=['eval_set', 'metric', 'value'])
    for tag, res in results.items():
        for k, v in res.items():
            if k in ('s', 'y'): continue
            table.add_data(tag, k, _cast(v))
    run.log({table_name: table})

    # log images
    run.log({f"plots/{n}": wandb.Image(f) for n, f in figs.items()})

# Model

### Aggregate Funcs

In [54]:
def aggregate_funcs(funcs):
  return np.array([
        np.concatenate([r['X'].mean(0), r['X'].sum(0), r['X'].max(0), r['X'].std(0), [len(r['X'])]])
        for r in funcs
    ], np.float64)


### Build positive pairs

In [55]:
def build_pair_index(labels):
  by_class = {}
  for i, c in enumerate(labels):
    by_class.setdefault(int(c), []).append(i)
  pair_classes = [c for c, v in by_class.items() if len(v) >= 2]
  return by_class, pair_classes

### Sample batch

In [56]:
def sample_batch(by_class, pair_classes, B, rng):
  cs = rng.choice(pair_classes, size=min(B, len(pair_classes)), replace=False)
  anc, pos = [], []
  for c in cs:
    i, j = rng.choice(by_class[c], 2, replace=False)
    anc.append(i); pos.append(j)
  return np.array(anc), np.array(pos)

### Pad batch

In [57]:
def pad_batch(funcs, device):
  mats = [torch.tensor(f['X'], dtype=torch.float32) for f in funcs]
  T = max(len(m) for m in mats)
  B, D = len(mats), mats[0].shape[1]
  X = torch.zeros(B, T, D)
  mask = torch.zeros(B, T, dtype=torch.bool)
  for i, m in enumerate(mats):
    X[i, :len(m)] = m
    mask[i, :len(m)] = True
  return X.to(device), mask.to(device)

### Loss infoNCE

In [58]:
def info_nce(anchors, positives, temperature=.05):
    a = F.normalize(anchors, dim=1)
    p = F.normalize(positives, dim=1)
    logits = a @ p.T / temperature
    labels = torch.arange(len(a), device=a.device)
    return F.cross_entropy(logits, labels)

### Loss TripletLoss

In [59]:
def triplet_loss(anchors, positives, margin=0.2, mining='batch_hard'):
  a = F.normalize(anchors, dim=1)
  p = F.normalize(positives, dim=1)
  sim = a @ p.T

  pos_sim = sim.diag()
  B = len(a)
  eye = torch.eye(B, dtype=torch.bool, device=a.device)
  neg_sim = sim.masked_fill(eye, -1e9)

  if mining == 'batch_hard':
    hardest_neg = neg_sim.max(dim=1).values
    loss = F.relu(margin + hardest_neg - pos_sim)
    return loss.mean()
  else:
    losses = F.relu(margin + neg_sim - pos_sim.unsqueeze(1))
    losses = losses.masked_fill(eye, 0.0)
    return losses.sum() / (B * (B - 1))

### Loss MSE

In [60]:
def mse_loss(anchors, positives, negatives_weight=1.0):
    a = F.normalize(anchors, dim=1)
    p = F.normalize(positives, dim=1)
    sim = a @ p.T
    B = len(a)
    target = torch.eye(B, device=a.device)
    return F.mse_loss(sim, target)

### Model Class

In [61]:
def init_mlp(in_dim, hiddens, out_dim, dropout=0.0):
    layers = []
    prev = in_dim
    for h in hiddens:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    layers.append(nn.Linear(prev, out_dim))
    return nn.Sequential(*layers)

In [62]:
class DeepSets(nn.Module):
    def __init__(self, in_dim, phi_hiddens, phi_out, rho_hiddens, out_dim,
                 pool='mean_max', dropout=0.0):
        super().__init__()
        self.pool_type = pool
        self.in_norm = nn.BatchNorm1d(in_dim)
        self.phi_net = init_mlp(in_dim, phi_hiddens, phi_out, dropout)

        if pool in ('simple_attention', 'mean_max_attention'):
            self.att = nn.Linear(phi_out, 1)

        n_views = {
            'mean': 1, 'max': 1, 'sum': 1, 'simple_attention': 1,
            'mean_max': 2,
            'mean_max_attention': 3,
            'kitchen_sink': 4,
        }[pool]

        pooled_dim = phi_out * n_views

        self.rho_net = init_mlp(pooled_dim, rho_hiddens, out_dim, dropout)


    def forward(self, X, mask):
        B, T, D = X.shape
        Xn = self.in_norm(X.reshape(-1, D)).reshape(B, T, D)
        h = self.phi_net(Xn)
        m = mask.unsqueeze(-1).float()

        def mean_pool():
            return (h * m).sum(1) / m.sum(1).clamp(min=1)

        def max_pool():
            return h.masked_fill(m == 0, -1e9).max(1).values

        def sum_pool():
            return (h * m).sum(1)

        def std_pool():
            mu = (h * m).sum(1) / m.sum(1).clamp(min=1)
            var = ((h - mu.unsqueeze(1))**2 * m).sum(1) / m.sum(1).clamp(min=1)
            return torch.sqrt(var + 1e-6)

        def attention_pool():
            scores = self.att(h).squeeze(-1)
            scores = scores.masked_fill(~mask, -1e9)
            weights = F.softmax(scores, dim=1).unsqueeze(-1)
            return (h * weights).sum(1)

        if self.pool_type == 'mean':
            pooled = mean_pool()
        elif self.pool_type == 'max':
            pooled = max_pool()
        elif self.pool_type == 'sum':
            pooled = sum_pool()
        elif self.pool_type == 'simple_attention':
            pooled = attention_pool()
        elif self.pool_type == 'mean_max':
            pooled = torch.cat([mean_pool(), max_pool()], dim=1)
        elif self.pool_type == 'mean_max_attention':
            pooled = torch.cat([mean_pool(), max_pool(), attention_pool()], dim=1)
        elif self.pool_type == 'kitchen_sink':
            pooled = torch.cat([mean_pool(), max_pool(), sum_pool(), std_pool()], dim=1)

        return self.rho_net(pooled)

In [63]:
# Adapter for core model
class DeepSetsEmbedder:
  def __init__(self, model, device='cpu', batch_size=256):
    self.model = model.to(device).eval()
    self.device = device
    self.batch_size = batch_size

  @torch.no_grad()
  def embed_batch(self, funcs):
    out = []
    for i in range(0, len(funcs), self.batch_size):
      batch = funcs[i:i+self.batch_size]
      X, mask = pad_batch(batch, self.device)
      out.append(self.model(X, mask).cpu().numpy())
    return np.concatenate(out, axis=0)


## Similarity Functions

### Cos similarity

In [64]:
def cos_sim(E, query_index, cand_index):
  En = _normalize(E)
  return np.dot(En[query_index], En[cand_index].T)

### L2 similarity

In [65]:
def l2_sim(E, query_index, cand_index):
  diff = E[query_index] - E[cand_index]
  return -np.linalg.norm(diff, axis=-1)

# Train Model

In [66]:
class _EarlyStopper:
    def __init__(self, patience):
        self.patience = patience
        self.best = -np.inf
        self.best_state = None
        self.waited = 0

    def update(self, metric, model):
        if metric > self.best:
            self.best = metric
            self.best_state = {k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()}
            self.waited = 0
            return False
        self.waited += 1
        return self.patience is not None and self.waited >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)

In [67]:
@torch.no_grad()
def embedding_health(ea, ep):
    an = F.normalize(ea, dim=1)
    pn = F.normalize(ep, dim=1)
    sim = an @ pn.T
    pos_sim = sim.diag().mean().item()
    neg_sim = ((sim.sum() - sim.diag().sum()) / (sim.numel() - len(sim))).item()
    return {'pos_sim': pos_sim, 'neg_sim': neg_sim, 'sim_gap': pos_sim - neg_sim}

def log_train(step, loss, ea, ep, lr):
    d = {'train/loss': loss, 'train/lr': lr}
    d.update({f'train/{k}': v for k, v in embedding_health(ea, ep).items()})
    wandb.log(d, step=step)


In [68]:
def get_loss_fn(loss_name, kwargs):
  if loss_name == 'info_nce':
      loss_fn = lambda ea, ep: info_nce(ea, ep, **kwargs)
  elif loss_name == 'triplet':
      loss_fn = lambda ea, ep: triplet_loss(ea, ep, **kwargs)
  return loss_fn


In [69]:
def train_step(model, opt, train_funcs, batch, loss_fn, device):
    anc, pos = batch
    Xa, ma = pad_batch([train_funcs[i] for i in anc], device)
    Xp, mp = pad_batch([train_funcs[i] for i in pos], device)
    ea, ep = model(Xa, ma), model(Xp, mp)
    loss = loss_fn(ea, ep)

    opt.zero_grad()
    loss.backward()
    opt.step()
    return loss.item(), (ea, ep)

In [70]:
def validate(model, val_fn, step, use_wandb):
    model.eval()
    vr = val_fn(model)
    model.train()
    if use_wandb:
        wandb.log({f'val/{k}': _cast(v) for k, v in vr.items()
                   if k not in ('s', 'y')}, step=step)
    return vr

In [71]:
def train_model(model, train_funcs, train_labels, loss_fn, steps=2000, batch_size=128,
                lr=1e-3, device='cpu', val_fn=None, val_every=100,
                seed=1337, use_wandb=False, early_stop_patience=None):
    by_class, pair_classes = build_pair_index(train_labels)

    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    rng = np.random.default_rng(seed)
    stopper = _EarlyStopper(early_stop_patience)

    model.train()
    for step in range(steps):
        batch = sample_batch(by_class, pair_classes, batch_size, rng)
        loss, (ea, ep) = train_step(model, opt, train_funcs, batch, loss_fn, device)

        if use_wandb:
            log_train(step, loss, ea, ep, opt.param_groups[0]['lr'])

        if step % val_every == 0 and val_fn:
            vr = validate(model, val_fn, step, use_wandb)
            print(f'step {step}: loss={loss:.4f}  val_auc={vr.get("roc_auc", 0):.4f}  '
                  f'val_R@1@1000={vr.get("recall@1_pool1000", float("nan")):.3f}')
            if stopper.update(vr.get('roc_auc', 0), model):
                print(f'early stop at step {step} (best val_auc={stopper.best:.4f})')
                break

    stopper.restore(model)
    model.eval()
    if use_wandb:
        wandb.run.summary['best_val_auc'] = stopper.best
    return model

# Init Datasets

In [96]:
# Train hyperparams
min_blocks    = 4
n_feat        = 21
score_fn      = cos_sim
pool_sizes    = (10, 100, 1000)
n_pairs       = 10000
n_queries     = 3000
ks            = (1, 5, 10)
per_axis      = True

# Read data
openssl_train_funcs, openssl_val_funcs, openssl_test_funcs, openssl_train_labels, openssl_val_labels, openssl_test_labels = make_funcs_labels(f'{PATH}/data_acfg/openssl_1_0_1f_acfg', min_blocks=min_blocks, n_feat=n_feat, ratios=(.8, .1, .1))
_,                   _,                 zlib_test_funcs,    _,                    _,                  zlib_test_labels    = make_funcs_labels(f'{PATH}/data_acfg/zlib_acfg',           min_blocks=min_blocks, n_feat=n_feat, ratios=(0, 0, 1))
_,                   _,                 sqlite_test_funcs,  _,                    _,                  sqlite_test_labels  = make_funcs_labels(f'{PATH}/data_acfg/sqlite3_acfg',        min_blocks=min_blocks, n_feat=n_feat, ratios=(0, 0, 1))

[load] files=12 funcs=40624 classes=4332 dropped(<4 blocks)=30344 n_feat=21
[split] train=32475 val=4074 test=4075
[load] files=12 funcs=3093 classes=368 dropped(<4 blocks)=941 n_feat=21
[split] train=0 val=0 test=3093
[load] files=12 funcs=19864 classes=3679 dropped(<4 blocks)=5773 n_feat=21
[split] train=0 val=0 test=19864


In [97]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# hyperparams
in_dim              = n_feat
phi_hiddens         = [128, 256]
phi_out             = 256
rho_hiddens         = [128]
out_dim             = 64
pool                = 'mean_max_attention'
steps               = 5000
batch_size          = 128
lr                  = 1e-3
dropout             = 0.0
seed                = 1337
val_every           = 100
early_stop_patience = 8

# Loss function
# loss_name       = 'info_nce'
# loss_params     = {'temperature': .05}

loss_name       = 'triplet'
loss_params     = {'margin': 1, 'mining': 'all'}
loss_fn         = get_loss_fn(loss_name, loss_params)


# Val function
def make_val_fn(val_funcs, val_labels, device):
    def val_fn(model):
        emb = DeepSetsEmbedder(model, device)
        return evaluate(emb, cos_sim, val_funcs, val_labels,
                        n_pairs=2000, n_queries=500, pool_sizes=(100,1000),
                        ks=(1,5,10), per_axis=True)
    return val_fn


# Run name
run_name = (f'deepsets'
            f'__pool_{pool}'
            f'__nfeat_{n_feat}__minblk_{min_blocks}'
            f'__phi_{"_".join(map(str, phi_hiddens))}_{phi_out}'
            f'__rho_{"_".join(map(str, rho_hiddens))}_{out_dim}'
            f'__lr_{lr}__dropout_{dropout}__bs_{batch_size}'
            f'__steps_{steps}__sim_{score_fn.__name__}'
            f'__loss_{loss_name}')

for k, v in loss_params.items():
    run_name += f'__{k}_{v}'


run = wandb.init(
    project='binsim_deepsets',
    name=run_name,
    config={
        'model': 'deepsets',
        'pool': pool,
        'in_dim': in_dim,
        'phi_hiddens': phi_hiddens,
        'phi_out': phi_out,
        'rho_hiddens': rho_hiddens,
        'out_dim': out_dim,
        'n_feat': n_feat,
        'min_blocks': min_blocks,
        'steps': steps,
        'batch_size': batch_size,
        'lr': lr,
        'dropout': dropout,
        'loss_name': loss_name,
        **loss_params,
        'seed': seed,
        'sim_fn': score_fn.__name__,
        'pool_sizes': pool_sizes,
        'n_pairs': n_pairs,
        'n_queries': n_queries,
        'ks': ks,
        'per_axis': per_axis,
        'arches': ['x86','arm32','mips32'],
        'opts': ['O0','O1','O2','O3'],
        'train_lib': 'openssl-1.0.1f',
        'eval_sets': ['openssl_1_0_1f','zlib-1.3.1','sqlite3'],
    }
)


# Train model
torch.manual_seed(seed)
model = DeepSets(in_dim, phi_hiddens, phi_out, rho_hiddens, out_dim,
                 pool=pool, dropout=dropout).to(device)
wandb.watch(model, log='all', log_freq=100)
val_fn = make_val_fn(openssl_val_funcs, openssl_val_labels, device)
model = train_model(model, openssl_train_funcs, openssl_train_labels, loss_fn,
                    steps=steps, batch_size=batch_size, lr=lr, device=device,
                    val_fn=val_fn, val_every=val_every, seed=seed,
                    use_wandb=True, early_stop_patience=early_stop_patience)

# Log Metrics of Trained Model
embedder = DeepSetsEmbedder(model, device)

tests = {
    'openssl_1_0_1f': (openssl_test_funcs, openssl_test_labels),
    'zlib-1.3.1':     (zlib_test_funcs,    zlib_test_labels),
    'sqlite3':        (sqlite_test_funcs,  sqlite_test_labels),
}

results = {}
for tag, (test, labels) in tests.items():
    results[tag] = evaluate(embedder, score_fn, test, labels,
                            pool_sizes=pool_sizes, n_pairs=n_pairs,
                            n_queries=n_queries, ks=ks, per_axis=per_axis)

print_comparison(results)

figs = make_plots(results, axes=['XA', 'XO', 'XM'], pools=pool_sizes, ks=ks,
                  model_name='deepsets')
log(run, results, figs)

# Log model artifact
model_path = 'deepsets.pt'
torch.save({'state_dict': model.state_dict(),
            'config': {'in_dim': in_dim, 'phi_hiddens': phi_hiddens, 'phi_out': phi_out,
                       'rho_hiddens': rho_hiddens, 'out_dim': out_dim, 'pool': pool}},
           'deepsets.pt')
artifact = wandb.Artifact('deepsets', type='model')
artifact.add_file('deepsets.pt')
run.log_artifact(artifact)
run.finish()

device: cuda


step 0: loss=0.9525  val_auc=0.7542  val_R@1@1000=0.124
step 100: loss=0.2852  val_auc=0.9524  val_R@1@1000=0.312
step 200: loss=0.2226  val_auc=0.9589  val_R@1@1000=0.324
step 300: loss=0.1985  val_auc=0.9631  val_R@1@1000=0.366
step 400: loss=0.1963  val_auc=0.9693  val_R@1@1000=0.338
step 500: loss=0.1661  val_auc=0.9714  val_R@1@1000=0.364
step 600: loss=0.1643  val_auc=0.9726  val_R@1@1000=0.400
step 700: loss=0.1974  val_auc=0.9774  val_R@1@1000=0.416
step 800: loss=0.1719  val_auc=0.9771  val_R@1@1000=0.388
step 900: loss=0.2453  val_auc=0.9795  val_R@1@1000=0.384
step 1000: loss=0.1731  val_auc=0.9779  val_R@1@1000=0.420
step 1100: loss=0.1360  val_auc=0.9776  val_R@1@1000=0.394
step 1200: loss=0.1979  val_auc=0.9803  val_R@1@1000=0.402
step 1300: loss=0.1506  val_auc=0.9810  val_R@1@1000=0.408
step 1400: loss=0.1314  val_auc=0.9815  val_R@1@1000=0.400
step 1500: loss=0.1421  val_auc=0.9826  val_R@1@1000=0.404
step 1600: loss=0.1476  val_auc=0.9814  val_R@1@1000=0.380
step 1700

openssl_1_0_1f/XA_mrr_pool10,▁
openssl_1_0_1f/XA_mrr_pool100,▁
openssl_1_0_1f/XA_mrr_pool1000,▁
openssl_1_0_1f/XA_recall@10_pool10,▁
openssl_1_0_1f/XA_recall@10_pool100,▁
openssl_1_0_1f/XA_recall@10_pool1000,▁
openssl_1_0_1f/XA_recall@1_pool10,▁
openssl_1_0_1f/XA_recall@1_pool100,▁
openssl_1_0_1f/XA_recall@1_pool1000,▁
openssl_1_0_1f/XA_recall@5_pool10,▁
+179,...


# Custom Test

In [74]:
def find_good_functions(funcs, labels, min_blocks=15, min_compilations=4, top_n=20):
    """Find substantial, well-represented functions worth testing on."""
    by_class = {}
    for i, c in enumerate(labels):
        by_class.setdefault(int(c), []).append(i)

    candidates = []
    for c, idxs in by_class.items():
        if len(idxs) < min_compilations:
            continue
        # average block count across this function's compilations
        avg_blocks = np.mean([funcs[i]['n_num'] for i in idxs])
        if avg_blocks < min_blocks:
            continue
        name = funcs[idxs[0]].get('fname', '?')
        candidates.append((c, name, avg_blocks, len(idxs)))

    # sort by size (biggest = most distinctive/interesting)
    candidates.sort(key=lambda x: -x[2])
    print(f"{'function':<35} {'avg_blocks':>10} {'#compilations':>14}")
    print("-" * 62)
    for c, name, blocks, n in candidates[:top_n]:
        print(f"{name[:35]:<35} {blocks:>10.0f} {n:>14}")
    return [c for c, _, _, _ in candidates[:top_n]]

good_classes = find_good_functions(openssl_test_funcs, openssl_test_labels,
                                   min_blocks=15, min_compilations=4, top_n=20)

function                            avg_blocks  #compilations
--------------------------------------------------------------
d2i_SSL_SESSION                            318             12
OpenSSL_add_all_ciphers                    208             12
BN_mod_sqrt                                204             12
X509_verify_cert                           178             12
ASN1_item_ex_d2i                           129             12
test_ecdh_curve                            128             12
tls1_change_cipher_state                   112             12
RSA_check_key                              108             12
SMIME_write_ASN1                           106             12
do_passwd                                  106             12
ssl_cipher_get_evp                         106             12
ec_GF2m_simple_add                         104             12
group_order_tests                          102             12
b64_read                                   102             12
BN_mod_

In [75]:
def retrieval_on_good(embedder, funcs, labels, good_classes, pool_size=1000, k=5, seed=0):
    """Retrieval demo restricted to substantial, distinctive functions."""
    rng = np.random.default_rng(seed)
    E = embedder.embed_batch(funcs)
    En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-10)
    arches = [f.get('arch') for f in funcs]

    by_class = {}
    for i, c in enumerate(labels):
        by_class.setdefault(int(c), []).append(i)

    total_hits, total_possible = 0, 0
    for c in good_classes:
        idxs = by_class[c]
        # query with x86, look for ARM/MIPS matches (hardest cross-arch case)
        x86s = [i for i in idxs if arches[i] == 'x86']
        others = set(idxs) - set(x86s)
        if not x86s or not others:
            continue
        q = x86s[0]

        distractors = rng.choice(len(funcs), pool_size, replace=False)
        pool = list((set(others) | set(distractors)) - {q})
        sims = En[q] @ En[pool].T
        order = np.argsort(-sims)[:k]

        hits = sum(1 for idx in order if pool[idx] in others)
        total_hits += hits
        total_possible += min(k, len(others))

        print(f"\nQUERY (x86): {funcs[q].get('fname','?')}  "
              f"[{funcs[q]['n_num']} blocks]")
        for rank, idx in enumerate(order, 1):
            fi = pool[idx]
            is_match = fi in others
            mark = f'✓ {arches[fi]}' if is_match else '  MISS'
            print(f"   #{rank} sim={sims[idx]:.3f} {mark:9} "
                  f"{funcs[fi].get('fname','?')} [{arches[fi]}]")

    print(f"\n{'='*50}")
    print(f"cross-arch hit rate on GOOD functions: "
          f"{total_hits}/{total_possible} = {total_hits/max(total_possible,1):.1%}")

retrieval_on_good(embedder, openssl_test_funcs, openssl_test_labels, good_classes)


QUERY (x86): d2i_SSL_SESSION  [328 blocks]
   #1 sim=0.948   MISS    d2i_SSL_SESSION [x86]
   #2 sim=0.927 ✓ arm32   d2i_SSL_SESSION [arm32]
   #3 sim=0.920 ✓ arm32   d2i_SSL_SESSION [arm32]
   #4 sim=0.919 ✓ arm32   d2i_SSL_SESSION [arm32]
   #5 sim=0.893 ✓ mips32  d2i_SSL_SESSION [mips32]

QUERY (x86): OpenSSL_add_all_ciphers  [209 blocks]
   #1 sim=0.984 ✓ arm32   OpenSSL_add_all_ciphers [arm32]
   #2 sim=0.976 ✓ mips32  OpenSSL_add_all_ciphers [mips32]
   #3 sim=0.974   MISS    OpenSSL_add_all_ciphers [x86]
   #4 sim=0.972 ✓ mips32  OpenSSL_add_all_ciphers [mips32]
   #5 sim=0.969 ✓ arm32   OpenSSL_add_all_ciphers [arm32]

QUERY (x86): BN_mod_sqrt  [236 blocks]
   #1 sim=0.990 ✓ arm32   BN_mod_sqrt [arm32]
   #2 sim=0.978   MISS    BN_mod_sqrt [x86]
   #3 sim=0.970 ✓ arm32   BN_mod_sqrt [arm32]
   #4 sim=0.968 ✓ mips32  BN_mod_sqrt [mips32]
   #5 sim=0.961 ✓ arm32   BN_mod_sqrt [arm32]

QUERY (x86): X509_verify_cert  [129 blocks]
   #1 sim=0.951 ✓ mips32  X509_verify_cert [mips32]

In [76]:
KNOWN_GOOD = [
    'AES_encrypt', 'AES_decrypt', 'AES_cbc_encrypt',
    'SHA1_Update', 'SHA256_Update', 'SHA512_Update',
    'BN_mul', 'BN_div', 'BN_mod_exp', 'BN_sqr',            # bignum — big & distinctive
    'RSA_public_encrypt', 'RSA_private_decrypt',
    'EC_POINT_add', 'EC_POINT_mul',
    'ssl3_accept', 'ssl3_connect',                          # TLS state machines — huge
    'dtls1_process_heartbeat',                              # Heartbleed!
    'EVP_EncryptUpdate', 'EVP_DecryptUpdate',
    'ASN1_get_object', 'asn1_ex_c2i',
]

def test_named_functions(embedder, funcs, labels, names, pool_size=1000, k=5, seed=0):
    """Test retrieval on specific named functions (if present in the data)."""
    rng = np.random.default_rng(seed)
    E = embedder.embed_batch(funcs)
    En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-10)
    arches = [f.get('arch') for f in funcs]

    # map name -> indices
    name_to_idx = {}
    for i, f in enumerate(funcs):
        name_to_idx.setdefault(f.get('fname'), []).append(i)

    for name in names:
        if name not in name_to_idx or len(name_to_idx[name]) < 2:
            continue                                        # not present or too few compilations
        idxs = name_to_idx[name]
        x86s = [i for i in idxs if arches[i] == 'x86']
        others = set(idxs) - set(x86s)
        if not x86s or not others:
            continue
        q = x86s[0]
        distractors = rng.choice(len(funcs), pool_size, replace=False)
        pool = list((set(others) | set(distractors)) - {q})
        sims = En[q] @ En[pool].T
        order = np.argsort(-sims)[:k]

        print(f"\nQUERY: {name}  [{funcs[q]['n_num']} blocks, x86]")
        for rank, idx in enumerate(order, 1):
            fi = pool[idx]
            mark = f'✓ {arches[fi]}' if fi in others else '  '
            print(f"   #{rank} sim={sims[idx]:.3f} {mark:8} "
                  f"{funcs[fi].get('fname','?')} [{arches[fi]}]")

test_named_functions(embedder, openssl_test_funcs, openssl_test_labels, KNOWN_GOOD)


QUERY: BN_sqr  [54 blocks, x86]
   #1 sim=0.963          BN_sqr [x86]
   #2 sim=0.927 ✓ arm32  BN_sqr [arm32]
   #3 sim=0.927 ✓ arm32  BN_sqr [arm32]
   #4 sim=0.918 ✓ arm32  BN_sqr [arm32]
   #5 sim=0.910 ✓ mips32 BN_sqr [mips32]


In [77]:
def list_available_good_functions(funcs, labels, candidates):
    """Show which known-good functions are in your dataset and how big."""
    name_to = {}
    for i, f in enumerate(funcs):
        name_to.setdefault(f.get('fname'), []).append(i)

    print(f"{'function':<35} {'present?':>8} {'blocks':>7} {'#comps':>7}")
    print("-" * 60)
    for name in candidates:
        if name in name_to:
            idxs = name_to[name]
            blocks = int(np.mean([funcs[i]['n_num'] for i in idxs]))
            print(f"{name[:35]:<35} {'YES':>8} {blocks:>7} {len(idxs):>7}")
        else:
            print(f"{name[:35]:<35} {'no':>8} {'-':>7} {'-':>7}")

list_available_good_functions(openssl_test_funcs, openssl_test_labels, KNOWN_GOOD)

function                            present?  blocks  #comps
------------------------------------------------------------
AES_encrypt                               no       -       -
AES_decrypt                               no       -       -
AES_cbc_encrypt                           no       -       -
SHA1_Update                               no       -       -
SHA256_Update                             no       -       -
SHA512_Update                             no       -       -
BN_mul                                    no       -       -
BN_div                                    no       -       -
BN_mod_exp                                no       -       -
BN_sqr                                   YES      46      12
RSA_public_encrypt                        no       -       -
RSA_private_decrypt                       no       -       -
EC_POINT_add                              no       -       -
EC_POINT_mul                              no       -       -
ssl3_accept             

In [78]:
import wandb, torch, numpy as np

def load_deepsets_from_wandb(artifact_path):
    """
    artifact_path like: 'sbolk23-free-university-of-tbilisi-/binsim_deepsets/deepsets:v0'
    """
    api = wandb.Api()
    artifact = api.artifact(artifact_path, type='model')
    artifact_dir = artifact.download()

    ckpt = torch.load(f'{artifact_dir}/deepsets.pt', map_location='cpu')
    cfg = ckpt['config']

    model = DeepSets(
        in_dim=cfg['in_dim'],
        phi_hiddens=cfg['phi_hiddens'],
        phi_out=cfg['phi_out'],
        rho_hiddens=cfg['rho_hiddens'],
        out_dim=cfg['out_dim'],
        pool=cfg['pool'],
    )
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    print(f"loaded DeepSets: pool={cfg['pool']}, out_dim={cfg['out_dim']}")
    return model

# load it
model = load_deepsets_from_wandb(
    'sbolk23-free-university-of-tbilisi-/binsim_deepsets/deepsets:v6'
)
embedder = DeepSetsEmbedder(model, device='cuda' if torch.cuda.is_available() else 'cpu')

wandb:   1 of 1 files downloaded.  


loaded DeepSets: pool=mean_max, out_dim=64


In [79]:
def retrieval_demo(embedder, funcs, labels, n_queries=5, pool_size=500, k=10, seed=10):
    """Pick random queries, show top-k retrieved functions. Mark which are TRUE matches."""
    rng = np.random.default_rng(seed)
    E = embedder.embed_batch(funcs)
    En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-10)

    by_class = {}
    for i, c in enumerate(labels):
        by_class.setdefault(int(c), []).append(i)
    multi = [c for c, v in by_class.items() if len(v) >= 2]

    for _ in range(n_queries):
        c = multi[rng.integers(len(multi))]
        q = by_class[c][0]
        true_matches = set(by_class[c]) - {q}

        distractors = rng.choice(len(funcs), pool_size, replace=False)
        pool = list(set(list(true_matches) + list(distractors)) - {q})

        sims = En[q] @ En[pool].T
        order = np.argsort(-sims)[:k]

        qname = funcs[q].get('fname', '?')
        qsrc = funcs[q].get('src', '?')
        print(f"\n{'='*70}")
        print(f"QUERY: {qname}  [{qsrc}]")
        print(f"{'='*70}")
        hits = 0
        for rank, idx in enumerate(order, 1):
            fi = pool[idx]
            is_match = fi in true_matches
            if is_match: hits += 1
            mark = '✓ MATCH' if is_match else '        '
            print(f"  #{rank:2}  sim={sims[idx]:.3f}  {mark}  "
                  f"{funcs[fi].get('fname','?')}  [{funcs[fi].get('src','?')}]")
        print(f"  --> {hits}/{len(true_matches)} true matches found in top-{k}")

retrieval_demo(embedder, openssl_test_funcs, openssl_test_labels,
               n_queries=5, pool_size=500, k=10)


QUERY: TS_TST_INFO_get_ext_d2i  [openssl_1_0_1f-arm32-O0/TS_TST_INFO_get_ext_d2i]
  # 1  sim=0.997            d2i_PROXY_POLICY  [openssl_1_0_1f-arm32-O0/d2i_PROXY_POLICY]
  # 2  sim=0.997            d2i_RSAPublicKey  [openssl_1_0_1f-arm32-O0/d2i_RSAPublicKey]
  # 3  sim=0.985            SEED_ofb128_encrypt  [openssl_1_0_1f-arm32-O1/SEED_ofb128_encrypt]
  # 4  sim=0.951            PKCS8_PRIV_KEY_INFO_free  [openssl_1_0_1f-arm32-O0/PKCS8_PRIV_KEY_INFO_free]
  # 5  sim=0.951            GENERAL_SUBTREE_free  [openssl_1_0_1f-arm32-O0/GENERAL_SUBTREE_free]
  # 6  sim=0.948            PEM_read_bio_X509_CERT_PAIR  [openssl_1_0_1f-mips32-O0/PEM_read_bio_X509_CERT_PAIR]
  # 7  sim=0.948            PEM_read_bio_DSA_PUBKEY  [openssl_1_0_1f-mips32-O0/PEM_read_bio_DSA_PUBKEY]
  # 8  sim=0.944            PEM_write_X509  [openssl_1_0_1f-mips32-O0/PEM_write_X509]
  # 9  sim=0.933  ✓ MATCH  TS_TST_INFO_get_ext_d2i  [openssl_1_0_1f-mips32-O0/TS_TST_INFO_get_ext_d2i]
  #10  sim=0.927            CMS_Recei

In [80]:
def tsne_demo(embedder, funcs, labels, n_functions=8, seed=110):
    """Visualize: do all compilations of the same function cluster together?"""
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt

    rng = np.random.default_rng(seed)
    by_class = {}
    for i, c in enumerate(labels):
        by_class.setdefault(int(c), []).append(i)
    multi = [c for c, v in by_class.items() if len(v) >= 12]      # need several compilations
    chosen = rng.choice(multi, min(n_functions, len(multi)), replace=False)

    idxs, cls = [], []
    for c in chosen:
        for i in by_class[c]:
            idxs.append(i); cls.append(c)

    sub_funcs = [funcs[i] for i in idxs]
    E = embedder.embed_batch(sub_funcs)
    coords = TSNE(n_components=2, perplexity=min(15, len(idxs)-1),
                  random_state=seed).fit_transform(E)

    plt.figure(figsize=(10, 8))
    for c in chosen:
        pts = [j for j, cc in enumerate(cls) if cc == c]
        name = sub_funcs[pts[0]].get('fname', str(c))[:20]
        plt.scatter(coords[pts, 0], coords[pts, 1], label=name, s=80, alpha=0.7)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.title(f't-SNE: each color = one function, points = its compilations\n'
              f'(tight clusters = model groups compilations of the same function)')
    plt.tight_layout()
    plt.savefig(f'/content/tsne_functions_v6_{seed}.png', dpi=120, bbox_inches='tight')
    plt.show()

tsne_demo(embedder,openssl_test_funcs, openssl_test_labels, n_functions=8, seed=11010)

In [81]:
def similarity_separation(embedder, funcs, labels, n=2000, seed=1):
    import matplotlib.pyplot as plt
    rng = np.random.default_rng(seed)
    E = embedder.embed_batch(funcs)
    En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-10)

    by_class = {}
    for i, c in enumerate(labels):
        by_class.setdefault(int(c), []).append(i)
    multi = [c for c, v in by_class.items() if len(v) >= 2]

    same, diff = [], []
    for _ in range(n):
        c = multi[rng.integers(len(multi))]
        i, j = rng.choice(by_class[c], 2, replace=False)
        same.append(En[i] @ En[j])
        a, b = rng.integers(len(funcs)), rng.integers(len(funcs))
        if labels[a] != labels[b]:
            diff.append(En[a] @ En[b])

    plt.figure(figsize=(9, 5))
    plt.hist(same, bins=50, alpha=0.6, label=f'same function (mean {np.mean(same):.2f})', density=True)
    plt.hist(diff, bins=50, alpha=0.6, label=f'different function (mean {np.mean(diff):.2f})', density=True)
    plt.xlabel('cosine similarity'); plt.ylabel('density')
    plt.title('Separation: same-function vs different-function similarity\n'
              '(more separation = better)')
    plt.legend()
    plt.savefig('/content/similarity_separation.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"same-function mean sim:      {np.mean(same):.3f}")
    print(f"different-function mean sim: {np.mean(diff):.3f}")
    print(f"separation gap:              {np.mean(same) - np.mean(diff):.3f}")

similarity_separation(embedder, openssl_test_funcs, openssl_test_labels)

same-function mean sim:      0.745
different-function mean sim: 0.155
separation gap:              0.590


In [82]:
def cross_arch_demo(embedder, funcs, labels, n_queries=5, pool_size=500, seed=0):
    rng = np.random.default_rng(seed)
    E = embedder.embed_batch(funcs)
    En = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-10)
    arches = [f.get('arch') for f in funcs]

    by_class = {}
    for i, c in enumerate(labels):
        by_class.setdefault(int(c), []).append(i)
    multi = [c for c, v in by_class.items() if len(v) >= 2]

    for _ in range(n_queries):
        c = multi[rng.integers(len(multi))]
        # pick an x86 query, look for its ARM/MIPS matches
        x86s = [i for i in by_class[c] if arches[i] == 'x86']
        others = [i for i in by_class[c] if arches[i] != 'x86']
        if not x86s or not others: continue
        q = x86s[0]

        distractors = rng.choice(len(funcs), pool_size, replace=False)
        pool = list((set(others) | set(distractors)) - {q})
        sims = En[q] @ En[pool].T
        order = np.argsort(-sims)[:5]

        print(f"\nQUERY (x86): {funcs[q].get('fname','?')}")
        print(f"  looking for its ARM/MIPS versions...")
        for rank, idx in enumerate(order, 1):
            fi = pool[idx]
            is_match = fi in set(others)
            mark = f'✓ {arches[fi]}' if is_match else '  '
            print(f"    #{rank} sim={sims[idx]:.3f} {mark:8} {funcs[fi].get('fname','?')} [{arches[fi]}]")

cross_arch_demo(embedder, openssl_test_funcs, openssl_test_labels)


QUERY (x86): X509_REVOKED_get_ext
  looking for its ARM/MIPS versions...
    #1 sim=1.000          TS_REQ_get_ext [x86]
    #2 sim=0.994          X509_STORE_CTX_set_flags [x86]
    #3 sim=0.983          i2d_OCSP_CERTSTATUS [x86]
    #4 sim=0.959          function_LHASH_HASH [x86]
    #5 sim=0.959          ENGINE_unregister_DH [x86]

QUERY (x86): i2d_PKCS7_ISSUER_AND_SERIAL
  looking for its ARM/MIPS versions...
    #1 sim=0.998          GENERAL_NAME_dup [x86]
    #2 sim=0.995          update512 [x86]
    #3 sim=0.983          test_sha1_final [x86]
    #4 sim=0.983          TS_REQ_delete_ext [x86]
    #5 sim=0.975          SSL_CTX_set_srp_client_pwd_callback [x86]

QUERY (x86): ENGINE_get_STORE
  looking for its ARM/MIPS versions...
    #1 sim=1.000          ENGINE_get_init_function [x86]
    #2 sim=0.997          X509_STORE_CTX_set_cert [x86]
    #3 sim=0.965          OPENSSL_stderr [x86]
    #4 sim=0.957          EVP_aes_128_cfb [arm32]
    #5 sim=0.941          TS_TST_INFO_get_ext_b

In [83]:
def tsne_specific_functions(embedder, funcs, labels, function_names,
                            color_by='function', seed=42):
    """
    Plot t-SNE for specific named functions (to compare with Gemini's plot).
    color_by='function': each function a different color (see clustering)
    color_by='arch':     color by architecture (see if archs mix or separate)
    """
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt

    # find all compilations of the requested functions
    name_to_idx = {}
    for i, f in enumerate(funcs):
        name_to_idx.setdefault(f.get('fname'), []).append(i)

    idxs, fnames, farches, fopts = [], [], [], []
    for name in function_names:
        if name not in name_to_idx:
            print(f"  WARNING: '{name}' not found in data")
            continue
        for i in name_to_idx[name]:
            idxs.append(i)
            fnames.append(name)
            farches.append(funcs[i].get('arch', '?'))
            fopts.append(funcs[i].get('opt', '?'))

    if not idxs:
        print("No functions found!")
        return

    sub_funcs = [funcs[i] for i in idxs]
    E = embedder.embed_batch(sub_funcs)
    coords = TSNE(n_components=2, perplexity=min(15, len(idxs)-1),
                  random_state=seed, init='pca').fit_transform(E)

    plt.figure(figsize=(11, 9))

    if color_by == 'function':
        # each function = one color, marker shape = architecture
        unique_names = list(dict.fromkeys(fnames))
        arch_markers = {'x86': 'o', 'arm32': 's', 'arm': 's', 'mips32': '^', 'mips': '^'}
        cmap = plt.cm.tab10
        for ni, name in enumerate(unique_names):
            for arch in set(farches):
                pts = [j for j in range(len(idxs))
                       if fnames[j] == name and farches[j] == arch]
                if pts:
                    plt.scatter(coords[pts, 0], coords[pts, 1],
                                color=cmap(ni % 10),
                                marker=arch_markers.get(arch, 'x'),
                                s=110, alpha=0.75, edgecolors='black', linewidths=0.5,
                                label=name if arch == farches[[j for j in range(len(idxs)) if fnames[j]==name][0]] else None)
        # legend: functions (colors) + architectures (markers)
        from matplotlib.lines import Line2D
        func_handles = [Line2D([0],[0], marker='o', color='w',
                               markerfacecolor=cmap(i % 10), markersize=11, label=n)
                        for i, n in enumerate(unique_names)]
        arch_handles = [Line2D([0],[0], marker=m, color='gray', linestyle='',
                               markersize=10, label=a)
                        for a, m in [('x86','o'), ('arm32','s'), ('mips32','^')]]
        leg1 = plt.legend(handles=func_handles, title='function',
                          bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
        plt.gca().add_artist(leg1)
        plt.legend(handles=arch_handles, title='architecture',
                   bbox_to_anchor=(1.02, 0.45), loc='upper left', fontsize=9)
        title = ('t-SNE of function embeddings\n'
                 'color = function, marker = architecture\n'
                 '(tight same-color clusters = model unifies compilations)')

    else:  # color_by == 'arch'
        arch_colors = {'x86':'tab:blue', 'arm32':'tab:orange', 'arm':'tab:orange',
                       'mips32':'tab:green', 'mips':'tab:green'}
        for arch in set(farches):
            pts = [j for j in range(len(idxs)) if farches[j] == arch]
            plt.scatter(coords[pts, 0], coords[pts, 1],
                        color=arch_colors.get(arch, 'gray'),
                        s=110, alpha=0.7, edgecolors='black', linewidths=0.5,
                        label=arch)
        plt.legend(title='architecture', bbox_to_anchor=(1.02, 1), loc='upper left')
        title = ('t-SNE colored by architecture\n'
                 '(if functions cluster regardless of arch color = arch-invariant embeddings)')

    plt.title(title)
    plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
    plt.tight_layout()
    plt.savefig(f'/content/tsne_{color_by}.png', dpi=130, bbox_inches='tight')
    plt.show()


# the specific functions you want (Gemini-style comparison)
FUNCTIONS = [
    'v2i_POLICY_MAPPINGS',
    # 'genrsa_main',
    'priv_decode_gost',
    'prompt_info',
    'ssl3_get_message',
]

# Plot 1: colored by function (see if compilations cluster)
tsne_specific_functions(embedder, funcs, labels,
                        FUNCTIONS, color_by='function')

# Plot 2: colored by architecture (see if the model is arch-invariant)
tsne_specific_functions(embedder, funcs, labels,
                        FUNCTIONS, color_by='arch')

NameError: name 'funcs' is not defined

In [ ]:
funcs, labels, classes = load_dataset('/content/Cross-Platform-Binary-Code-Similarity-Detection/data_acfg/openssl_1_0_1f_acfg', min_blocks=min_blocks, n_feat=n_feat)


name_to = {}
for i, f in enumerate(funcs):
    name_to.setdefault(f.get('fname'), []).append(i)
for fn in FUNCTIONS:
    n = len(name_to.get(fn, []))
    print(f"{fn}: {n} compilations {'✓' if n >= 2 else '✗ (not enough / in train split)'}")